In [2]:
import numpy as np


#Add file path
AP = r"C:\Users\wongbe\Desktop\PROJECTS_PC\npix_data_bhw\2026-10-02_15-08-00_NPIX_MSIT_014\Record Node 123\experiment1\recording1\continuous\OneBox-122.ProbeA-AP"
N_CH = 384
F0 = 27_660_000   # put the index you want to start at, in this case segment start time that we put through Kilosort4 is 922 s x 30000 frames per second = 27,660,000 frames
N = 1000          # frames to compare between raw and segment files just to make sure we are aligned

# memmap = look at the file on disk without loading all 37 GB into RAM
raw = np.memmap(AP + r"\continuous.dat", dtype="int16", mode="r").reshape(-1, N_CH)
seg = np.memmap(AP + r"\seg_g060\continuous.dat", dtype="int16", mode="r").reshape(-1, N_CH)
L = seg.shape[0]
print("raw frames:", raw.shape[0], "| segment frames:", L)

print("start matches at F0:     ", np.array_equal(raw[F0:F0+N], seg[:N]))
print("end matches at F0+L:     ", np.array_equal(raw[F0+L-N:F0+L], seg[L-N:]))
print("start matches at F0-1000:", np.array_equal(raw[F0-1000:F0-1000+N], seg[:N]))
print("start matches at F0+1:   ", np.array_equal(raw[F0+1:F0+1+N], seg[:N]))

# we want the output to be True, True, False, False, which means the segment file is aligned with the raw file at F0 and F0+L, but not at F0-1000 or F0+1.

raw frames: 48159900 | segment frames: 18660000
start matches at F0:      True
end matches at F0+L:      True
start matches at F0-1000: False
start matches at F0+1:    False


In [ ]:
import os

AP = r"C:\Users\wongbe\Desktop\PROJECTS_PC\npix_data_bhw\2026-10-02_15-08-00_NPIX_MSIT_014\Record Node 123\experiment1\recording1\continuous\OneBox-122.ProbeA-AP"
OUT = AP + r"\seg_g060\interpseg_g060\kilosort4\timestamps.npy"
F0 = 27_660_000
L = os.path.getsize(AP + r"\seg_g060\continuous.dat") // (384 * 2)   # segment length in frames

ts = np.load(AP + r"\timestamps.npy")
print("full timestamps:", ts.shape, ts.dtype, "| first value:", ts[0])

seg_ts = ts[F0:F0 + L]
print("slice length:", len(seg_ts), "(should be", L, ")")
print("slice start minus recording start (s):", seg_ts[0] - ts[0], "(expect ~922)")
print("slice duration (s):", seg_ts[-1] - seg_ts[0], "(expect ~622)")

assert len(seg_ts) == L
assert not os.path.exists(OUT), "timestamps.npy already exists, not overwriting"
np.save(OUT, seg_ts)
print("saved:", OUT)

#output:
# full timestamps: (file should have one time value per sample above), first value is when the recorder clock started in the OR
# slice length: the piece we cut should be as long as the segment we want from above
# slice start minus recording start: should output time in seconds after the FIRST entry, which is when our segment starts
# slice duration: should be the segment length 

full timestamps: (48159900,) float64 | first value: 72.2996
slice length: 18660000 (should be 18660000 )
slice start minus recording start (s): 922.0000000000196 (expect ~922)
slice duration (s): 621.9999666665977 (expect ~622)
saved: C:\Users\wongbe\Desktop\PROJECTS_PC\npix_data_bhw\2026-10-02_15-08-00_NPIX_MSIT_014\Record Node 123\experiment1\recording1\continuous\OneBox-122.ProbeA-AP\seg_g060\interpseg_g060\kilosort4\timestamps.npy
